# NB4: Aceleración en GPU con CuPy

**Computación de Altas Prestaciones para Ciencia de Datos (CAPCD)**

---

## Objetivos de este notebook

1. Entender la arquitectura de una GPU y por qué es tan rápida para datos.
2. Comprender la transferencia de datos Host (CPU) ↔ Device (GPU).
3. Usar **CuPy** como *drop-in replacement* de NumPy en GPU.
4. Saber cuándo la GPU es más rápida y cuándo no.

**Requisito**: Asegúrate de tener GPU activada en el runtime de Colab.

In [ ]:
# Verificar GPU
!nvidia-smi
!pip install -q cupy-cuda12x

---

## 1. CPU vs GPU: arquitectura

### CPU (Central Processing Unit)
- **Pocos núcleos potentes** (2-64 típicamente).
- Optimizada para tareas secuenciales complejas (branch prediction, out-of-order execution).
- Gran caché por núcleo.

### GPU (Graphics Processing Unit)
- **Miles de núcleos simples** (una T4 tiene 2560 CUDA cores).
- Optimizada para la misma operación sobre muchos datos (SIMT: *Single Instruction, Multiple Threads*).
- Altísimo ancho de banda de memoria (~300 GB/s en T4 vs ~50 GB/s en CPU).

### ¿Cuándo gana la GPU?

- Operaciones elemento a elemento sobre arrays grandes (> 100K elementos).
- Álgebra lineal (multiplicación de matrices).
- Reduciones (sumas, medias) sobre datos grandes.

### ¿Cuándo pierde la GPU?

- Datos pequeños (overhead de transferencia > ahorro).
- Código con muchas ramas (divergencia) (if/else complejos).
- Operaciones secuenciales (paso N depende de paso N-1).

---

## 2. El cuello de botella real: transferencia de datos

```
  CPU (Host)                 GPU (Device)
┌──────────┐   PCIe bus   ┌──────────────┐
│  RAM     │ ◄──────────► │  VRAM (GPU)  │
│ (Sistema)│  ~12 GB/s    │  (16 GB T4)  │
└──────────┘              └──────────────┘
```

La conexión CPU ↔ GPU (PCIe) es mucho más lenta que la memoria interna de la GPU.

**Regla fundamental:** minimiza las transferencias. Mueve los datos a la GPU una vez, haz todos los cálculos allí, y trae el resultado al final.

## Demo 1: Coste de la transferencia

In [ ]:
import numpy as np
import cupy as cp
import time

N = 10_000_000

# Crear datos en CPU
data_cpu = np.random.rand(N).astype(np.float32)

# Medir transferencia CPU → GPU
start = time.perf_counter()
data_gpu = cp.asarray(data_cpu)
cp.cuda.Stream.null.synchronize()  # Esperar a que termine la transferencia
t_to_gpu = time.perf_counter() - start

# Medir transferencia GPU → CPU
start = time.perf_counter()
data_back = cp.asnumpy(data_gpu)
t_to_cpu = time.perf_counter() - start

size_mb = data_cpu.nbytes / 1e6
print(f"Tamaño del array: {size_mb:.1f} MB")
print(f"CPU → GPU: {t_to_gpu*1000:.1f} ms  ({size_mb/t_to_gpu/1000:.1f} GB/s)")
print(f"GPU → CPU: {t_to_cpu*1000:.1f} ms  ({size_mb/t_to_cpu/1000:.1f} GB/s)")
print(f"\n⚠️ Si el cálculo tarda menos que la transferencia, la GPU no vale la pena.")

---

## 3. CuPy: NumPy en la GPU

**CuPy** implementa la misma API que NumPy, pero ejecuta las operaciones en la GPU.

```python
import numpy as np          # CPU
import cupy as cp           # GPU

# NumPy (CPU)
a = np.random.rand(1000000)
b = np.sum(a ** 2)

# CuPy (GPU) — ¡mismo código!
a = cp.random.rand(1000000)
b = cp.sum(a ** 2)
```

### Conversiones

| Operación | Código |
|---|---|
| NumPy → CuPy (CPU→GPU) | `gpu_array = cp.asarray(numpy_array)` |
| CuPy → NumPy (GPU→CPU) | `numpy_array = cp.asnumpy(gpu_array)` |
| Crear directo en GPU | `gpu_array = cp.random.rand(N)` |

## Demo 2: CuPy vs NumPy en operaciones elemento a elemento

In [ ]:
import numpy as np
import cupy as cp
import time

N = 50_000_000

# --- NumPy (CPU) ---
a_cpu = np.random.rand(N).astype(np.float32)
b_cpu = np.random.rand(N).astype(np.float32)

start = time.perf_counter()
c_cpu = np.sqrt(a_cpu ** 2 + b_cpu ** 2)  # Distancia al origen
t_numpy = time.perf_counter() - start

# --- CuPy (GPU) ---
a_gpu = cp.asarray(a_cpu)
b_gpu = cp.asarray(b_cpu)

# Warmup
_ = cp.sqrt(a_gpu ** 2 + b_gpu ** 2)
cp.cuda.Stream.null.synchronize()

start = time.perf_counter()
c_gpu = cp.sqrt(a_gpu ** 2 + b_gpu ** 2)
cp.cuda.Stream.null.synchronize()  # Importante: GPU es asíncrona
t_cupy = time.perf_counter() - start

# Verificar resultado
c_gpu_cpu = cp.asnumpy(c_gpu)
print(f"NumPy (CPU): {t_numpy*1000:.1f} ms")
print(f"CuPy  (GPU): {t_cupy*1000:.1f} ms")
print(f"Speedup:     {t_numpy/t_cupy:.1f}x")
print(f"¿Iguales?:   {np.allclose(c_cpu, c_gpu_cpu, rtol=1e-5)}")

## Demo 3: Multiplicación de matrices, la aplicación perfecta para la GPU

In [ ]:
N = 4096

# NumPy
A_cpu = np.random.rand(N, N).astype(np.float32)
B_cpu = np.random.rand(N, N).astype(np.float32)

start = time.perf_counter()
C_cpu = A_cpu @ B_cpu  # Multiplicación de matrices
t_numpy = time.perf_counter() - start

# CuPy
A_gpu = cp.asarray(A_cpu)
B_gpu = cp.asarray(B_cpu)
_ = A_gpu @ B_gpu  # Warmup
cp.cuda.Stream.null.synchronize()

start = time.perf_counter()
C_gpu = A_gpu @ B_gpu
cp.cuda.Stream.null.synchronize()
t_cupy = time.perf_counter() - start

print(f"Matrices {N}x{N} (float32)")
print(f"NumPy (CPU): {t_numpy:.3f}s")
print(f"CuPy  (GPU): {t_cupy:.3f}s")
print(f"Speedup:     {t_numpy/t_cupy:.0f}x")

## Demo 4: Cuando la GPU NO vale la pena (datos pequeños)

In [ ]:
print("Tamaño del array vs speedup GPU:\n")

for N in [100, 1_000, 10_000, 100_000, 1_000_000, 10_000_000]:
    a_cpu = np.random.rand(N).astype(np.float32)
    a_gpu = cp.asarray(a_cpu)

    # Warmup GPU
    cp.sum(a_gpu ** 2)
    cp.cuda.Stream.null.synchronize()

    # NumPy
    start = time.perf_counter()
    for _ in range(100):
        np.sum(a_cpu ** 2)
    t_numpy = (time.perf_counter() - start) / 100

    # CuPy
    start = time.perf_counter()
    for _ in range(100):
        cp.sum(a_gpu ** 2)
        cp.cuda.Stream.null.synchronize()
    t_cupy = (time.perf_counter() - start) / 100

    speedup = t_numpy / t_cupy
    ganador = "GPU ✅" if speedup > 1 else "CPU ✅"
    print(f"  N={N:>12,}  →  NumPy: {t_numpy*1000:.3f}ms  CuPy: {t_cupy*1000:.3f}ms  "
          f"Speedup: {speedup:.1f}x  {ganador}")


### Conclusión

Para arrays pequeños (< ~10,000 elementos), el *overhead* de lanzar el kernel en GPU domina. La GPU empieza a ganar a partir de ~100K elementos.

---

## Demo 5: Pipeline completo con datos persistentes en GPU

La clave del rendimiento es encadenar operaciones en GPU sin traer datos a la CPU entre medias.


In [ ]:
N = 100_000_000

# === Pipeline NumPy (CPU) ===
data_cpu = np.random.rand(N).astype(np.float32)

start = time.perf_counter()
# Normalizar
data_norm_cpu = (data_cpu - np.mean(data_cpu)) / np.std(data_cpu)
# Aplicar función no lineal
data_nl_cpu = np.tanh(data_norm_cpu)
# Calcular histograma
hist_cpu, _ = np.histogram(data_nl_cpu, bins=100)
t_numpy = time.perf_counter() - start

# === Pipeline CuPy (GPU) -- datos ya en GPU ===
data_gpu = cp.asarray(data_cpu)  # Una sola transferencia
cp.cuda.Stream.null.synchronize()

# Warmup
_ = cp.tanh((data_gpu - cp.mean(data_gpu)) / cp.std(data_gpu))
cp.cuda.Stream.null.synchronize()

start = time.perf_counter()
# Todo en GPU, sin mover datos
data_norm_gpu = (data_gpu - cp.mean(data_gpu)) / cp.std(data_gpu)
data_nl_gpu = cp.tanh(data_norm_gpu)
hist_gpu = cp.histogram(data_nl_gpu, bins=100)
cp.cuda.Stream.null.synchronize()
t_cupy = time.perf_counter() - start

print(f"Pipeline NumPy (CPU): {t_numpy*1000:.1f} ms")
print(f"Pipeline CuPy  (GPU): {t_cupy*1000:.1f} ms")
print(f"Speedup: {t_numpy/t_cupy:.1f}x")

---

## Demo 6: Impacto de la precisión numérica en GPU

Las GPUs fueron diseñadas para gráficos, donde la precisión aproximada es aceptable. Esto tiene una consecuencia importante para ciencia de datos:

| Tipo | Bits | Rango | Throughput relativo GPU |
|---|---|---|---|
| `float64` (double) | 64 | ±1.8×10³⁰⁸ | 1× (más lento) |
| `float32` (single) | 32 | ±3.4×10³⁸ | **~2×** |
| `float16` (half) | 16 | ±65,504 | **~4×** |

**¿Por qué?** En cada ciclo de reloj, el bus de la GPU transfiere un bloque fijo de bits. Con `float32` caben el doble de números que con `float64`, y con `float16` el cuádruple.

En **CPU**, `float16` es *más lento* porque no hay instrucciones nativas. En GPU es al revés.

Para muchos problemas de ciencia de datos (ML, estadística, procesamiento de señales), `float32` es suficiente.

In [ ]:
import cupy as cp
import time

N = 4096

print(f"Multiplicación de matrices {N}×{N} — impacto de la precisión:\n")

for dtype_name, dtype in [("float64", cp.float64), ("float32", cp.float32), ("float16", cp.float16)]:
    A = cp.random.rand(N, N).astype(dtype)
    B = cp.random.rand(N, N).astype(dtype)

    # Warmup
    _ = A @ B
    cp.cuda.Stream.null.synchronize()

    start = time.perf_counter()
    for _ in range(5):
        C = A @ B
        cp.cuda.Stream.null.synchronize()
    t = (time.perf_counter() - start) / 5

    print(f"  {dtype_name:8s} → {t*1000:7.2f} ms  (memoria por matriz: {A.nbytes/1e6:.0f} MB)")

print("\n★ Misma operación, mismo tamaño, diferente precisión → gran diferencia de rendimiento.")
print("  float32 suele ser el 'sweet spot' para ciencia de datos en GPU.")

---

## Ejercicio 1: Tu primer cálculo en GPU

Calcula la **norma euclídea** de un vector grande con CuPy y compáralo con NumPy.

$$\|v\| = \sqrt{\sum_{i=1}^{n} v_i^2}$$

In [ ]:
import numpy as np
import cupy as cp
import time

N = 50_000_000
v_cpu = np.random.rand(N).astype(np.float32)

# NumPy
start = time.perf_counter()
norma_cpu = np.sqrt(np.sum(v_cpu ** 2))
t_numpy = time.perf_counter() - start

# TODO: CuPy
# 1. Transfiere v_cpu a la GPU con cp.asarray()
# 2. Calcula la norma usando cp.sqrt() y cp.sum()
# 3. No olvides cp.cuda.Stream.null.synchronize() después del cálculo
# 4. Convierte el resultado a CPU con .item() o float()

norma_gpu = None  # TODO: reemplaza
t_cupy = None     # TODO: mide el tiempo

print(f"NumPy: norma={norma_cpu:.4f}  ({t_numpy*1000:.1f} ms)")
if norma_gpu is not None and t_cupy is not None:
    print(f"CuPy:  norma={float(norma_gpu):.4f}  ({t_cupy*1000:.1f} ms)")
    print(f"Speedup: {t_numpy/t_cupy:.1f}x")
else:
    print("⚠️  Implementa la sección CuPy (reemplaza los None).")


In [ ]:
# Autoevaluación
assert norma_gpu is not None, "Debes calcular la norma con CuPy"
assert abs(float(norma_cpu) - float(norma_gpu)) < 1.0, "Los resultados deben ser similares"
print("✅ ¡Correcto! La norma GPU coincide con CPU.")

---

## Ejercicio 2: K-Nearest Neighbors bruto en GPU

Dado un conjunto de puntos y un punto query, encuentra los K vecinos más cercanos calculando todas las distancias (fuerza bruta). Este patrón es fundamental en *Machine Learning*.

Implementa con CuPy y compara con NumPy.

In [ ]:
import numpy as np
import cupy as cp
import time

N_PUNTOS = 1_000_000
N_DIMS = 128
K = 10

# Datos
puntos_cpu = np.random.rand(N_PUNTOS, N_DIMS).astype(np.float32)
query_cpu = np.random.rand(N_DIMS).astype(np.float32)


def knn_numpy(puntos, query, k):
    """K-NN por fuerza bruta en CPU."""
    distancias = np.sqrt(np.sum((puntos - query) ** 2, axis=1))
    indices = np.argpartition(distancias, k)[:k]
    return indices, distancias[indices]


def knn_cupy(puntos, query, k):
    """K-NN por fuerza bruta en GPU."""
    # TODO: Implementa lo mismo que knn_numpy pero con CuPy (cp.)
    # Pista: cambia np.* por cp.*
    return None, None  # TODO: reemplaza con la implementación real


# NumPy
start = time.perf_counter()
idx_cpu, dist_cpu = knn_numpy(puntos_cpu, query_cpu, K)
t_numpy = time.perf_counter() - start

# CuPy
puntos_gpu = cp.asarray(puntos_cpu)
query_gpu = cp.asarray(query_cpu)
# Warmup
_ = knn_cupy(puntos_gpu, query_gpu, K)
cp.cuda.Stream.null.synchronize()

start = time.perf_counter()
idx_gpu, dist_gpu = knn_cupy(puntos_gpu, query_gpu, K)
cp.cuda.Stream.null.synchronize()
t_cupy = time.perf_counter() - start

print(f"NumPy: {t_numpy*1000:.1f} ms")
if idx_gpu is not None:
    print(f"CuPy:  {t_cupy*1000:.1f} ms")
    print(f"Speedup: {t_numpy/t_cupy:.1f}x")
else:
    print("⚠️  Implementa knn_cupy (reemplaza el return None, None).")


In [ ]:
# Autoevaluación
assert idx_gpu is not None, "knn_cupy debe devolver índices"
assert len(cp.asnumpy(idx_gpu)) == K, f"Debe devolver {K} vecinos"
print("✅ ¡Correcto! K-NN funciona en GPU.")

---

## Ejercicio 3: Pipeline de normalización en GPU

Tienes una matriz de features (como en ML). Normaliza cada columna a media 0 y desviación 1 (**z-score normalization**), todo en GPU. Mide el speedup contra NumPy.

In [ ]:
N_SAMPLES = 2_000_000
N_FEATURES = 200

X_cpu = np.random.rand(N_SAMPLES, N_FEATURES).astype(np.float32)


def normalizar_numpy(X):
    """Z-score normalization por columna en CPU."""
    media = np.mean(X, axis=0)
    std = np.std(X, axis=0)
    return (X - media) / std


def normalizar_cupy(X):
    """Z-score normalization por columna en GPU."""
    # TODO: Implementa usando CuPy (cp.)
    # Calcula media y std por columna (axis=0)
    # Devuelve (X - media) / std
    return None  # TODO: reemplaza con la implementación real


# NumPy
start = time.perf_counter()
X_norm_cpu = normalizar_numpy(X_cpu)
t_numpy = time.perf_counter() - start

# CuPy
X_gpu = cp.asarray(X_cpu)
_ = normalizar_cupy(X_gpu)  # Warmup
cp.cuda.Stream.null.synchronize()

start = time.perf_counter()
X_norm_gpu = normalizar_cupy(X_gpu)
cp.cuda.Stream.null.synchronize()
t_cupy = time.perf_counter() - start

print(f"NumPy: {t_numpy*1000:.1f} ms")
if X_norm_gpu is not None:
    print(f"CuPy:  {t_cupy*1000:.1f} ms")
    print(f"Speedup: {t_numpy/t_cupy:.1f}x")
else:
    print("⚠️  Implementa normalizar_cupy (reemplaza el return None).")


In [ ]:
# Autoevaluación
assert X_norm_gpu is not None, "normalizar_cupy debe devolver el array normalizado, no None"
X_norm_gpu_cpu = cp.asnumpy(X_norm_gpu)
assert np.allclose(X_norm_cpu, X_norm_gpu_cpu, atol=1e-4), "Los resultados deben coincidir"
# Verificar que está normalizado: media ≈ 0, std ≈ 1
assert np.allclose(np.mean(X_norm_gpu_cpu, axis=0), 0, atol=1e-3), "La media debe ser ~0"
assert np.allclose(np.std(X_norm_gpu_cpu, axis=0), 1, atol=1e-3), "La std debe ser ~1"
print("✅ ¡Correcto! Normalización GPU funciona perfectamente.")


---

## Resumen

| Concepto | Detalle |
|---|---|
| **GPU vs CPU** | GPU: miles de núcleos simples, ideal para datos masivos |
| **Transferencia** | CPU ↔ GPU es lenta (~12 GB/s). Minimizar movimientos |
| **CuPy** | Misma API que NumPy, ejecuta en GPU |
| **`cp.asarray()`** | CPU → GPU |
| **`cp.asnumpy()`** | GPU → CPU |
| **Synchronize** | `cp.cuda.Stream.null.synchronize()` para medir tiempos reales |

### Cuándo usar CuPy

- Arrays > 100K elementos.
- Pipelines con muchas operaciones encadenadas (mantener datos en GPU).
- Álgebra lineal, estadística, FFT sobre datos grandes.

### Siguiente paso

CuPy es potente pero está limitado a operaciones que ya existen en su API. En el **NB5** veremos cómo escribir *kernels* personalizados con Numba `@cuda.jit` para cuando necesitemos lógica a medida en la GPU.